# M1-B1 — Exploration de l'historique des réservations

Ce notebook sert aux tâches 1 à 3 : connexion à la base, exploration en SQL, chargement des données.
Crée d'autres notebooks pour la suite (par exemple `02_modelisation.ipynb`).

In [3]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline plus tard
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

## Tâche 1 — La base répond-elle ?

In [4]:
pd.read_sql(text("SELECT COUNT(*) AS nb_reservations FROM reservations"), engine)

,nb_reservations
0,78703


## Tâche 2 — Explorer en SQL

Les calculs se font **dans la requête SQL** ; pandas sert seulement à afficher le résultat.

Pour commencer : les tables disponibles et leurs colonnes.

In [5]:
pd.read_sql(text("SELECT * FROM hotels"), engine)

,hotel_id,code,nom,type,ville
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve


In [6]:
pd.read_sql(text("SELECT * FROM reservations LIMIT 5"), engine)

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,1,1,0,6,2015,July,27,1,0,2,...,6,None,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01
1,2,1,1,88,2015,July,27,1,0,4,...,9,None,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01
2,3,1,1,65,2015,July,27,1,0,4,...,9,None,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01
3,4,1,1,92,2015,July,27,1,2,4,...,9,None,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01
4,5,1,1,257,2015,July,27,1,0,2,...,6,None,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01


### Réservations et taux d'annulation par hôtel

In [9]:
query = text("""
   select h.nom, h.code, count(r.reservation_id) as nb_reservation, AVG(r.is_canceled) as taux_annulation, ROUND(100 * AVG(r.is_canceled), 1) AS taux_annulation_pct from reservations r left join hotels h on r.hotel_id=h.hotel_id group by h.hotel_id
""")
pd.read_sql(query, engine)

,nom,code,nb_reservation,taux_annulation,taux_annulation_pct
0,Escale Atlantique Océan,Resort Hotel,26881,0.262937,26.3
1,Escale Atlantique Centre,City Hotel,51822,0.413164,41.3


### Taux d'annulation selon le mois d'arrivée

In [10]:
query = text("""
   SELECT EXTRACT(MONTH FROM r.arrival_date) AS mois_num, r.arrival_date_month, COUNT(*) AS nb_reservation, ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct FROM reservations r GROUP BY 1, 2 ORDER BY 1;
""")
pd.read_sql(query, engine)

,mois_num,arrival_date_month,nb_reservation,taux_pct
0,1.0,January,2248,24.8
1,2.0,February,3891,34.4
2,3.0,March,4824,30.6
3,4.0,April,5428,38.0
4,5.0,May,5478,35.0
5,6.0,June,5292,39.6
6,7.0,July,7348,37.5
7,8.0,August,8952,38.2
8,9.0,September,10508,39.2
9,10.0,October,11160,38.0


### Taux d'annulation selon le délai entre réservation et arrivée

In [11]:
query = text("""
   SELECT CASE
         WHEN r.lead_time <= 7   THEN '0-7 j'
         WHEN r.lead_time <= 30  THEN '8-30 j'
         WHEN r.lead_time <= 90  THEN '31-90 j'
         WHEN r.lead_time <= 180 THEN '91-180 j'
         WHEN r.lead_time <= 365 THEN '181-365 j'
         ELSE 'plus de 365 j'
       END AS decoupage,
       COUNT(*) AS nb_reservation,
       ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct
FROM reservations r
GROUP BY 1
ORDER BY MIN(r.lead_time);
""")
pd.read_sql(query, engine)

,decoupage,nb_reservation,taux_pct
0,0-7 j,13925,10.0
1,8-30 j,12602,26.5
2,31-90 j,20140,35.5
3,91-180 j,15748,43.4
4,181-365 j,14504,57.7
5,plus de 365 j,1784,77.6


### Taux d'annulation par segment de clientèle et par type d'acompte

In [15]:
query = text("""
    SELECT r.market_segment,
           COUNT(*) AS nb_reservation,
           ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct
    FROM reservations r
    GROUP BY 1
    ORDER BY taux_pct DESC
""")
pd.read_sql(query, engine)


,market_segment,nb_reservation,taux_pct
0,Undefined,2,100.0
1,Groups,13957,62.3
2,Offline TA/TO,18552,34.5
3,Online TA,33826,33.9
4,Aviation,127,22.8
5,Corporate,3733,15.9
6,Direct,7977,15.3
7,Complementary,529,11.7


In [14]:
query = text("""
    SELECT r.deposit_type,
           COUNT(*) AS nb_reservation,
           ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct
    FROM reservations r
    GROUP BY 1
    ORDER BY taux_pct DESC
""")
pd.read_sql(query, engine)

,deposit_type,nb_reservation,taux_pct
0,Non Refund,10461,99.2
1,No Deposit,68107,26.6
2,Refundable,135,7.4


### Rapport d'étonnement

*Note ici tes observations.*

Le City Hotel annule 41 % de ses réservations, le Resort 26 %. Le City fait aussi les deux tiers du volume. Un modèle entraîné sur les deux apprendra surtout le City.

Je me suis fait avoir sur les volumes par mois. Octobre sort à 11 160 réservations, janvier à 2 248. Ce n'est pas la saison. L'historique va de juillet 2015 à décembre 2016, donc juillet à décembre sont comptés deux fois et janvier à juin une seule. Les taux se comparent, les volumes non.

Le délai marche bien. 10 % d'annulation à moins d'une semaine avant l'arrivée, 43 % entre trois et six mois.

L'acompte, je ne comprends pas. Non Refund annule à 99,2 %. Un acompte non remboursable devrait retenir les gens, c'est l'inverse qui se passe. Ça tient sur 10 461 réservations et c'est pareil dans tous les segments : 96,1 % en Online TA, 99,9 % en Offline TA/TO.

En croisant acompte et délai, je revois ma conclusion sur le délai. Au-delà d'un an d'avance, 78 % d'annulation toutes réservations confondues, mais 24 % si je ne garde que les No Deposit. Une partie de l'effet du délai venait de l'acompte.

Deux catégories trop petites pour en tirer quoi que ce soit : le segment Undefined avec 2 réservations, l'acompte Refundable avec 135.

In [16]:
query = text("""
    SELECT r.deposit_type,
           CASE WHEN r.lead_time <= 7   THEN '0-7 j'
                WHEN r.lead_time <= 90  THEN '8-90 j'
                WHEN r.lead_time <= 365 THEN '91-365 j'
                ELSE 'plus de 365 j' END AS tranche_delai,
           COUNT(*) AS nb_reservation,
           ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct
    FROM reservations r
    GROUP BY 1, 2
    ORDER BY 1, MIN(r.lead_time)
""")
pd.read_sql(query, engine)

,deposit_type,tranche_delai,nb_reservation,taux_pct
0,No Deposit,0-7 j,13833,9.5
1,No Deposit,8-90 j,31005,28.4
2,No Deposit,91-365 j,22747,34.5
3,No Deposit,plus de 365 j,522,23.9
4,Non Refund,0-7 j,87,82.8
5,Non Refund,8-90 j,1710,99.0
6,Non Refund,91-365 j,7405,99.3
7,Non Refund,plus de 365 j,1259,100.0
8,Refundable,0-7 j,5,0.0
9,Refundable,8-90 j,27,0.0


In [17]:
query = text("""
    SELECT r.market_segment,
           COUNT(*) AS nb_reservation,
           ROUND(100 * AVG(r.is_canceled), 1) AS taux_pct
    FROM reservations r
    WHERE r.deposit_type = 'Non Refund'
    GROUP BY 1
    ORDER BY nb_reservation DESC
""")
pd.read_sql(query, engine)

,market_segment,nb_reservation,taux_pct
0,Groups,6395,99.1
1,Offline TA/TO,3856,99.9
2,Corporate,146,84.9
3,Online TA,51,96.1
4,Direct,13,76.9


## Tâche 3 — Charger les données et préparer le découpage

Une seule requête SQL, avec la jointure sur `hotels` pour récupérer la colonne `hotel`
(`'City Hotel'` / `'Resort Hotel'`) attendue par la recette de la v1.

In [ ]:
query = text("""
    SELECT r.*, h.code AS hotel
    FROM reservations r
    JOIN hotels h ON r.hotel_id = h.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date", "reservation_status_date"])
print(df.shape)
df.head()

(78703, 35)


,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,hotel
0,1,1,0,6,2015,July,27,1,0,2,...,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01,City Hotel
1,2,1,1,88,2015,July,27,1,0,4,...,NaN,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01,City Hotel
2,3,1,1,65,2015,July,27,1,0,4,...,NaN,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01,City Hotel
3,4,1,1,92,2015,July,27,1,2,4,...,NaN,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01,City Hotel
4,5,1,1,257,2015,July,27,1,0,2,...,NaN,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01,City Hotel


In [19]:
TEST_MONTHS = 4

# le test commence au 1er jour du 4e mois avant la fin de l'historique
dernier_mois = df["arrival_date"].max().to_period("M")
debut_test = (dernier_mois - (TEST_MONTHS - 1)).to_timestamp()

train = df[df["arrival_date"] < debut_test].copy()
test = df[df["arrival_date"] >= debut_test].copy()

print(f"Bascule au {debut_test:%Y-%m-%d}")
print(f"train : {len(train):>6} | {train['arrival_date'].min():%Y-%m-%d} → {train['arrival_date'].max():%Y-%m-%d} | {train['is_canceled'].mean():.1%} annulées")
print(f"test  : {len(test):>6} | {test['arrival_date'].min():%Y-%m-%d} → {test['arrival_date'].max():%Y-%m-%d} | {test['is_canceled'].mean():.1%} annulées")

Bascule au 2016-09-01
train :  58792 | 2015-07-01 → 2016-08-31 | 35.6% annulées
test  :  19911 | 2016-09-01 → 2016-12-31 | 38.0% annulées
